In [1]:
import pickle
import sys
import copy
import time
import os

import cobra
import sympy
import pandas as pd
import numpy as np

import multiprocessing
import gc

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params
from macromolecules.macromolecule import Macromolecule
from expression.build_me_model import flatten_list
from utils.parameters import human_model as m_model
from core.model import load_pickled_model

No objective coefficients in model. Unclear what should be optimized


In [2]:
mu_val = 1e-9
n_cores = 5
counter = 0

lp_path = '/data2/hratch/human_me/other/test_lp/'
tme0 = load_pickled_model(lp_path + 'working_version_' + str(counter) + '.pickle')

In [58]:
tme0 = load_pickled_model(lp_path + 'working_version_' + str(counter) + '.pickle')

def _add_boundary(metabs_, type_ = 'sink', tme_new = None):
    '''Metabs_ is a list of cobra.Metabolite or metabolite IDs'''
    if tme_new is None:
        tme_new = load_pickled_model(lp_path + 'working_version_' + str(counter) + '.pickle')
    if type(metabs_) != list:
        metabs_ = list(metabs_)
    
    with func.HiddenPrints():
        for m in tqdm(metabs_):
            if isinstance(m, cobra.Metabolite): # object
                tme_new.add_boundary(tme_new.metabolites.get_by_id(m.id), type =type_)
            else: # string
                tme_new.add_boundary(tme_new.metabolites.get_by_id(m), type =type_)

    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    return tme_new, sln, stat

def _add_boundary_2(m_ids, mu_val = 1e-9):
    '''Adds a list of m_ids manually as sinks, and solves'''
    
    tme1 = tme0.copy()
    m_ids = [m.id for m in tme1.metabolites]
    
    model_reaction_ids = [r.id for r in tme1.reactions]
    sinks = list()
    for m_id in tqdm(m_ids):
        r = cobra.Reaction('SK_' + m_id)
        r.add_metabolites({tme1.metabolites.get_by_id(m_id): -1})
        r._lower_bound = -1000
        r._upper_bound = 1000
        if r.id not in model_reaction_ids:
            sinks.append(r)

    print('Add reactions to model')
    tme1.add_reactions(sinks)
    print('solve')
    sln, stat, _ = tme1.solve_lp(mu_val = mu_val)



    reaction_ids = [r.id for r in tme1.reactions if 'biomass' in r.id] 
    res = pd.DataFrame(index = reaction_ids)
    res['reaction_index'] = pd.Series(res.index).apply(lambda x: tme1.reactions.index(x)).tolist()
    res['flux'] = res.reaction_index.apply(lambda x: sln[x])
    return tme1, sln, stat, res


In [37]:
# # params
# mu_val = 1e-9 





Current status:

In [ ]:
# INFEASIBLE
# biomass metabolites as sinks (very weird this is infeasible)
# only metabolic model metabolites
# only rna macromolecules
# just enzymes
# translation products only
# non-enzyme, non-translation product proteins

# FEASIBLE
# all metabolites except biomass 
# all macromolecules
# just proteins and complexes
# just proteins
# just proteins that are not enzymes - (no one individual protein makes model feasible)

# TRY NEXT: just complexes...

In [94]:
# params
mu_val = 1e-9 

tme1 = tme0.copy()
m_ids = set([m.id for m in tme1.metabolites if 'biomass' not in m.id \
            and hasattr(m, 'type') and m.type == 'protein' and not m.enzyme]) 
# # translational products only
# translation_ids = flatten_list([[m.id for m in r.products if hasattr(m, 'type') and m.type == 'protein'] for r in tme1.reactions if hasattr(r, 'translation') and r.translation])
# m_ids = set(m_ids).difference(translation_ids)


model_reaction_ids = [r.id for r in tme1.reactions]
sinks = list()
for m_id in tqdm(m_ids):
    r = cobra.Reaction('SK_' + m_id)
    r.add_metabolites({tme1.metabolites.get_by_id(m_id): -1})
    r._lower_bound = -1000
    r._upper_bound = 1000
    if r.id not in model_reaction_ids:
        sinks.append(r)

print('Add reactions to model')
tme1.add_reactions(sinks)
print('solve')
sln1, stat1, _ = tme1.solve_lp(mu_val = 1e-9)

100%|██████████| 3285/3285 [00:01<00:00, 2935.67it/s]


Add reactions to model
solve
Getting MINOS parameters...
Done in 53.5945 seconds with status 0


In [103]:
sln1, stat1, _ = tme1.solve_lp(mu_val = 0.01)
sln0, stat0, _ = tme0.solve_lp(mu_val = 0.01)

Getting MINOS parameters...
Done in 63.6999 seconds with status 0
Getting MINOS parameters...
Done in 69.5028 seconds with status 1


In [104]:
reaction_ids = set([r.id for r in tme0.reactions]).intersection([r.id for r in tme1.reactions])
res = pd.DataFrame(index = reaction_ids)

res['flux_feasible'] = pd.Series(res.index).apply(lambda x: tme1.reactions.index(x)).apply(lambda x: sln1[x]).tolist()
res['flux_infeasible'] = pd.Series(res.index).apply(lambda x: tme0.reactions.index(x)).apply(lambda x: sln0[x]).tolist()
res['flux_diff'] = (res.flux_feasible - res.flux_infeasible).abs()

biomass_ids = [r.id for r in tme0.reactions if 'biomass' in r.id]


# ir0 = tme0.infeasible_reactions(mu_val, sln0, stat0, tolerance = 0)
# ires = res.loc[ir0.keys(),:]

# ires['difference'] = (ires.flux_feasible - ires.flux_infeasible).abs()
# ires.sort_values(by = 'difference', ascending = False, inplace = True)

In [105]:
res.loc[biomass_ids, :]

,flux_feasible,flux_infeasible,flux_diff
biomass_dilution,1.000000e-02,1.000000e-02,0.000000e+00
DNA_biomass_to_biomass,1.400000e-04,1.400000e-04,0.000000e+00
carbohydrate_biomass_to_biomass,7.100000e-04,7.100000e-04,0.000000e+00
lipid_biomass_to_biomass,9.700000e-04,9.700000e-04,0.000000e+00
tRNA_biomass_to_biomass,0.000000e+00,0.000000e+00,0.000000e+00
rRNA_biomass_to_biomass,0.000000e+00,0.000000e+00,0.000000e+00
mRNA_biomass_to_biomass,-2.404910e-53,-1.501318e-38,1.501318e-38
premRNA_biomass_to_biomass,0.000000e+00,0.000000e+00,0.000000e+00
other_RNA_biomass_to_biomass,-8.524719e-45,-3.750374e-04,3.750374e-04
protein_biomass_to_biomass,8.180000e-03,8.555037e-03,3.750374e-04


In [108]:
tme0.metabolites.get_by_id('biomass_tRNA')

Metabolite identifier,biomass_tRNA
Name,
Memory address,0x07fd87b7a3a90
Formula,None
Compartment,None
In 1245 reaction(s),"HGNC:12628_TRANSLATION_ELONGATIONc, HGNC:21298_TRANSLATION_ELONGATIONc, HGNC:2520_TRANSLATION_ELONGATIONc, HGNC:7683_TRANSLATION_ELONGATIONc, HGNC:28977_co_TRANSLOC_IMPORTtr, HGNC:4816_TRANSLATION_..."


In [109]:
tme0.metabolites.get_by_id('biomass_tRNA').reactions

frozenset({<Protein_Expression_Reaction HGNC:12628_TRANSLATION_ELONGATIONc at 0x7fd8796300b8>,
           <Protein_Expression_Reaction HGNC:21298_TRANSLATION_ELONGATIONc at 0x7fd87a8100f0>,
           <Protein_Expression_Reaction HGNC:2520_TRANSLATION_ELONGATIONc at 0x7fd87a6000f0>,
           <Protein_Expression_Reaction HGNC:7683_TRANSLATION_ELONGATIONc at 0x7fd879d70128>,
           <Protein_Expression_Reaction HGNC:28977_co_TRANSLOC_IMPORTtr at 0x7fd878d10128>,
           <Protein_Expression_Reaction HGNC:4816_TRANSLATION_ELONGATIONc at 0x7fd878a70128>,
           <Protein_Expression_Reaction HGNC:10416_TRANSLATION_ELONGATIONc at 0x7fd87b510198>,
           <Protein_Expression_Reaction HGNC:381_TRANSLATION_ELONGATIONc at 0x7fd87a2c01d0>,
           <Protein_Expression_Reaction HGNC:4982_TRANSLATION_ELONGATIONc at 0x7fd87a020208>,
           <Protein_Expression_Reaction HGNC:17350_TRANSLATION_ELONGATIONc at 0x7fd879320208>,
           <Protein_Expression_Reaction HGNC:23208_TRANSLAT

In [111]:
tme0.reactions.get_by_id('HGNC:12628_TRANSLATION_ELONGATIONc').metabolites

{<Protein HGNC:12628_unfolded_protein_c at 0x7fd8796300f0>: 1,
 <Ribosomal_Complex TRANSLATION_ELONGATIONc_complex_c at 0x7fd87b5343c8>: -2.77012100054047e-6*mu - 2.82176554384009e-8,
 <tRNA charged_generic_A_trna_c at 0x7fd87b74c5c0>: -58,
 <tRNA charged_generic_R_trna_c at 0x7fd87b74c860>: -42,
 <tRNA charged_generic_N_trna_c at 0x7fd87b74cac8>: -24,
 <tRNA charged_generic_D_trna_c at 0x7fd87b74cd30>: -61,
 <tRNA charged_generic_C_trna_c at 0x7fd87b74cf98>: -15,
 <tRNA charged_generic_E_trna_c at 0x7fd87b75e240>: -70,
 <tRNA charged_generic_Q_trna_c at 0x7fd87b75e4a8>: -33,
 <tRNA charged_generic_G_trna_c at 0x7fd87b75e710>: -59,
 <tRNA charged_generic_H_trna_c at 0x7fd87b75e978>: -16,
 <tRNA charged_generic_I_trna_c at 0x7fd87b75ebe0>: -39,
 <tRNA charged_generic_L_trna_c at 0x7fd87b75ee48>: -71,
 <tRNA charged_generic_K_trna_c at 0x7fd87b6f10f0>: -51,
 <tRNA charged_generic_M_trna_c at 0x7fd87b6f1358>: -24,
 <tRNA charged_generic_F_trna_c at 0x7fd87b6f15c0>: -34,
 <tRNA charged_gen

In [113]:
tme0.reactions.get_by_id('CHARGING_TRNA_generic_A').metabolites

{<tRNA charged_generic_A_trna_c at 0x7fd87b74c5c0>: 1,
 <Protein HGNC:20_folded_protein_c at 0x7fd87b74c5f8>: -4.30402820763474e-5*mu - 7.59719104662557e-7,
 <Proxy HGNC:20_enzyme_degradation_proxy_c at 0x7fd87b74c828>: -7.597191046625572e-07,
 <tRNA generic_trna_c at 0x7fd87b734fd0>: -1,
 <Metabolite ala_L_c at 0x7fd87c2f8390>: -1,
 <Metabolite atp_c at 0x7fd87c3e3b00>: -1,
 <Metabolite ppi_c at 0x7fd87c3eb588>: 1,
 <Metabolite amp_c at 0x7fd87c3eb358>: 1}

In [73]:
res.status.unique()

array([1])

# parallelize above code

In [ ]:
import multiprocessing
import gc
n_cores = 17

tme0 = load_pickled_model(lp_path + 'working_version_' + str(counter) + '.pickle')

In [61]:
def par_sinks(m_id):
    '''Add m_id and solve'''
    
    tme1 = tme0.copy()
    
    model_reaction_ids = [r.id for r in tme1.reactions]
    
    sinks = list()
    r = cobra.Reaction('SK_' + m_id)
    r.add_metabolites({tme1.metabolites.get_by_id(m_id): -1})
    r._lower_bound = -1000
    r._upper_bound = 1000
    if r.id not in model_reaction_ids:
        sinks.append(r)

    tme1.add_reactions(sinks)
    sln, stat, _ = tme1.solve_lp(mu_val = mu_val)
    
    fn = '/data2/hratch/human_me/other/test_lp/test_sinks.tab'
    if not os.path.isfile(fn):
        with open(fn, 'a+') as f:
            f.write('metabolite_id' + '\t' + 'status' + '\n')
        
        
    with open(fn, 'a+') as f:
        f.write(m_id + '\t' + str(stat.max()) + '\n')

def add_boundary(m, type_ = 'sink', tme_new = None):
    '''m is a cobra.metabolite or metabolite ID'''
    
    if tme_new is None:
        tme_new = load_pickled_model(lp_path + 'working_version_' + str(counter) + '.pickle')
    
    if isinstance(m, cobra.Metabolite): # object
        m_id = m.id
    else: # string
        m_id = m
    
    tme_new.add_boundary(tme_new.metabolites.get_by_id(m_id), type = type_)
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    
    fn = '/data2/hratch/human_me/other/test_lp/test_' + type_ + '.tab'
    if not os.path.isfile(fn):
        with open(fn, 'a+') as f:
            f.write('metabolite_id' + '\t' + 'status' + '\n')
        
        
    with open(fn, 'a+') as f:
        f.write(m_id + '\t' + str(stat.max()) + '\n')

In [ ]:
print('Start parallelization')
pool = multiprocessing.Pool(processes = n_cores)
try:
    res = pool.map(par_sinks, m_ids)
    pool.close()
    pool.join()
    gc.collect()
except:
    pool.close()
    pool.join()
    gc.collect()
    raise ValueError('Parallelization failed')    

# print('Start sinks')
# pool = multiprocessing.Pool(processes = n_cores)
# try:
#     res = pool.starmap(add_boundary, zip(metabs_, ['sink']*len(metabs_)))
#     pool.close()
#     pool.join()
#     gc.collect()
# except:
#     pool.close()
#     pool.join()
#     gc.collect()
#     raise ValueError('Parallelization failed')                       